# nvidia_cutlass_dsl — smoke test

## Example 1 — the version and the CUDA version it targets (no GPU)

In [ ]:
import importlib.metadata as im
import cutlass

dist = im.distribution("nvidia-cutlass-dsl")
assert cutlass.__version__ == dist.version, (cutlass.__version__, dist.version)
assert cutlass.CUDA_VERSION.major >= 12, cutlass.CUDA_VERSION
print(f"cutlass {cutlass.__version__}, CUDA {cutlass.CUDA_VERSION.major}.{cutlass.CUDA_VERSION.minor}")

## Example 2 — CuTe layout algebra, traced and executed on the host (no GPU)

A `@cute.jit` function with no kernel runs entirely on the host: layouts are built in the closed MLIR core and evaluated at trace time. A row-major 4×8 layout maps `(2, 3)` to `2·8 + 3 = 19`, a compact column-major one coalesces to `32:1`, and `zipped_divide` by a 2×4 tile gives `((2, 4), (2, 2))`.

In [ ]:
import cutlass, cutlass.cute as cute

seen = {}

@cute.jit
def layout_algebra():
    row_major = cute.make_layout((4, 8), stride=(8, 1))
    seen["size"], seen["cosize"] = cute.size(row_major), cute.cosize(row_major)
    seen["idx"] = row_major((2, 3))
    seen["tiled"] = str(cute.zipped_divide(row_major, (2, 4)).shape)
    seen["coalesced"] = str(cute.coalesce(cute.make_layout((4, 8))))
    # row_major o (8:4) visits column-major indices 0,4,...,28 = row 0 across the 8 columns -> the identity 8:1
    seen["composed"] = str(cute.composition(row_major, cute.make_layout(8, stride=4)))

layout_algebra()
assert seen["size"] == 32 and seen["cosize"] == 32, seen
assert seen["idx"] == 19, seen
assert seen["tiled"] == "((2, 4), (2, 2))", seen
assert seen["coalesced"] == "32:1" and seen["composed"] == "8:1", seen
print("OK -- layout algebra traced and evaluated on the host:", seen)

## Example 3 — a `@cute.kernel` elementwise add launched on the GPU

In [ ]:
import torch, cutlass, cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

assert torch.cuda.is_available(), f"requires-gpu cell ran but torch {torch.__version__} sees no CUDA device"

@cute.kernel
def add_kernel(gA: cute.Tensor, gB: cute.Tensor, gC: cute.Tensor):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    m, n = gA.shape
    i = bidx * 256 + tidx
    gC[i // n, i % n] = gA[i // n, i % n] + gB[i // n, i % n]

@cute.jit
def add(mA: cute.Tensor, mB: cute.Tensor, mC: cute.Tensor):
    m, n = mA.shape
    add_kernel(mA, mB, mC).launch(grid=((m * n) // 256, 1, 1), block=(256, 1, 1))

cc = torch.cuda.get_device_capability()
if cc < (8, 0):
    print(f"SKIP: compute capability {cc[0]}.{cc[1]} is below the CuTe DSL's 8.0 floor (its Arch enum starts at Ampere) -- no JIT target")
else:
    a, b = (torch.randn(256, 512, device="cuda", dtype=torch.float16) for _ in range(2))
    c = torch.zeros_like(a)
    args = [from_dlpack(t, assumed_align=16) for t in (a, b, c)]
    cute.compile(add, *args)(*args)
    torch.cuda.synchronize()
    torch.testing.assert_close(c, a + b, rtol=0, atol=0)
    print(f"OK -- @cute.kernel elementwise add on {torch.cuda.get_device_name(0)} (sm_{cc[0]}{cc[1]}): bit-exact vs torch")

## Example 4 — a warp-level `mma.sync` GEMM (`MmaF16BF16Op`, sm_80+)

In [ ]:
import torch, cutlass, cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

@cute.kernel
def mma_kernel(gA: cute.Tensor, gB: cute.Tensor, gC: cute.Tensor, tiled_mma: cute.TiledMma):
    thr = tiled_mma.get_slice(cute.arch.thread_idx()[0])
    tCgA, tCgB, tCgC = thr.partition_A(gA), thr.partition_B(gB), thr.partition_C(gC)
    tCrA, tCrB, tCrC = tiled_mma.make_fragment_A(tCgA), tiled_mma.make_fragment_B(tCgB), tiled_mma.make_fragment_C(tCgC)
    cute.autovec_copy(tCgA, tCrA); cute.autovec_copy(tCgB, tCrB); tCrC.fill(0.0)
    cute.gemm(tiled_mma, tCrC, tCrA, tCrB, tCrC)
    cute.autovec_copy(tCrC, tCgC)

@cute.jit
def mma(mA: cute.Tensor, mB: cute.Tensor, mC: cute.Tensor):
    op = cute.nvgpu.warp.MmaF16BF16Op(cutlass.Float16, cutlass.Float32, (16, 8, 16))
    mma_kernel(mA, mB, mC, cute.make_tiled_mma(op)).launch(grid=(1, 1, 1), block=(32, 1, 1))

# get_device_capability() RAISES without a CUDA device -- a loud defect in a requires-gpu cell, never a skip.
cc = torch.cuda.get_device_capability()
if cc < (8, 0):
    print(f"SKIP: compute capability {cc[0]}.{cc[1]} has no F16 mma.sync.m16n8k16 (Ampere+) and is below the CuTe DSL's 8.0 floor")
else:
    # A is (M, K) and B is (N, K), both K-major (the op's fixed operand layout), so CuTe computes C = A @ B^T.
    a, b = (torch.randn(shape, device="cuda", dtype=torch.float16) for shape in ((16, 16), (8, 16)))
    c = torch.zeros(16, 8, device="cuda", dtype=torch.float32)
    args = [from_dlpack(t) for t in (a, b, c)]
    cute.compile(mma, *args)(*args)
    err = (c - a.float() @ b.float().T).abs().max().item()  # .item() synchronizes the stream
    assert c.abs().sum().item() > 0 and err < 1e-2, f"warp MMA GEMM mismatch vs torch: max |err| {err}"
    print(f"OK -- warp-level MmaF16BF16Op 16x8x16 GEMM on sm_{cc[0]}{cc[1]}: max |err| vs torch {err:.2e}")